# Round 3 — Return correlations, surface, lead–lag

Builds a wide mid-price panel, computes per-tick returns, correlation matrix, and cross-correlations (VELVET vs VEV) to summarize **how products move together** in the historical dump. This supports intuition for a ladder/surface, not a trading rule.

In [ ]:
from __future__ import annotations

import itertools
from pathlib import Path

import numpy as np
import pandas as pd


def _find_r3() -> Path:
    for base in (Path("."), Path(".."), Path("../..")):
        p = base.resolve()
        if (p / "prices_round_3_day_0.csv").is_file() or (p / "round3" / "prices_round_3_day_0.csv").is_file():
            return p
    raise FileNotFoundError("Could not find prices_round_3_day_0.csv — run Jupyter from ROUND3/ or ROUND3/notebooks/.")


R3 = _find_r3()


def price_csv(day: int) -> Path:
    for p in (R3 / f"prices_round_3_day_{day}.csv", R3 / "round3" / f"prices_round_3_day_{day}.csv"):
        if p.is_file():
            return p
    raise FileNotFoundError(f"No prices CSV for day {day} under {R3}")

In [ ]:
DAY = 0
df = pd.read_csv(price_csv(DAY), sep=";")
w = df.pivot_table(index="timestamp", columns="product", values="mid_price", aggfunc="first")
r = w.pct_change().replace([np.inf, -np.inf], np.nan)
print("Panel:", w.shape, "  valid return obs (example VEV_5000):", r["VEV_5000"].notna().sum())

In [ ]:
vol_1e4 = r.std() * 1e4  # rough "basis-point-like" scale for pct returns
vol_1e4.sort_values(ascending=False).to_frame("vol_1e4_x_pctchg_std")

In [ ]:
C = r.corr()
vev = sorted([c for c in C.columns if c.startswith("VEV_")], key=lambda s: int(s.split("_", 1)[1]))
C.loc[vev, vev].style.background_gradient(cmap="Blues", vmin=0, vmax=1).format(precision=3)

In [ ]:
if "VELVETFRUIT_EXTRACT" in r.columns and "HYDROGEL_PACK" in r.columns:
    for tgt in ("VEV_5000", "VEV_5100", "VEV_4000"):
        if tgt in r.columns:
            print(
                f"VELVET vs {tgt}:  corr = {r['VELVETFRUIT_EXTRACT'].corr(r[tgt]):.4f}"
            )
    print(
        f"VELVET vs HYDROGEL:  corr = {r['VELVETFRUIT_EXTRACT'].corr(r['HYDROGEL_PACK']):.4f}"
    )

In [ ]:
a, b = r["VELVETFRUIT_EXTRACT"], r["VEV_5000"]
print("VELVET vs VEV_5000 cross-correlation (lag = ticks; negative = VELVET leads)")
for lag in range(-5, 6):
    if lag == 0:
        cc = a.corr(b)
    elif lag < 0:
        cc = a.corr(b.shift(-lag))
    else:
        cc = a.shift(lag).corr(b)
    print(f"  lag {lag:3d}  {cc if np.isfinite(cc) else float('nan'):.5f}")

In [ ]:
pairs: list[tuple[float, str, str]] = []
for x, y in itertools.combinations(r.columns, 2):
    v = C.loc[x, y]
    if np.isfinite(v):
        pairs.append((float(v), x, y))
pairs.sort(reverse=True)
print("Top 12 correlations:")
for v, x, y in pairs[:12]:
    print(f"  {v:.3f}  {x:24s}  {y}")